In [1]:
!pip install gymnasium stable-baselines3 sb3-contrib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 4.6 MB/s eta 0:00:00


In [2]:
# !pip install gymnasium stable-baselines3 sb3-contrib torch scipy matplotlib pandas

import os
import csv
import time
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from scipy.ndimage import binary_dilation
from collections import deque

from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, CallbackList
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.monitor import Monitor
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.policies import MaskableActorCriticCnnPolicy

# =====================================================================
# 1. THÔNG SỐ & LUẬT GAME
# =====================================================================
BOARD_SIZE = 40
WIN_LEN    = 5
MAX_MOVES  = 600
DIRECTIONS = [(1,0),(0,1),(1,1),(1,-1)]

def check_win(board, r, c, player):
    for dr, dc in DIRECTIONS:
        cnt = 1
        nr, nc = r+dr, c+dc
        while 0<=nr<BOARD_SIZE and 0<=nc<BOARD_SIZE and board[nr,nc]==player:
            cnt+=1; nr+=dr; nc+=dc
        nr, nc = r-dr, c-dc
        while 0<=nr<BOARD_SIZE and 0<=nc<BOARD_SIZE and board[nr,nc]==player:
            cnt+=1; nr-=dr; nc-=dc
        if cnt >= WIN_LEN: return True
    return False

def static_evaluate(board, player, active_cells):
    score = 0.0
    for r, c in active_cells:
        p = board[r, c]
        if p == 0: continue
        mine = (p == player)
        for dr, dc in DIRECTIONS:
            pr, pc = r-dr, c-dc
            if 0<=pr<BOARD_SIZE and 0<=pc<BOARD_SIZE and board[pr,pc]==p:
                continue
            cnt = 1; nr, nc = r+dr, c+dc
            while 0<=nr<BOARD_SIZE and 0<=nc<BOARD_SIZE and board[nr,nc]==p:
                cnt+=1; nr+=dr; nc+=dc
            ends = 0
            if 0<=pr<BOARD_SIZE and 0<=pc<BOARD_SIZE and board[pr,pc]==0: ends+=1
            if 0<=nr<BOARD_SIZE and 0<=nc<BOARD_SIZE and board[nr,nc]==0: ends+=1
            if cnt >= WIN_LEN:   base = 200000
            elif ends == 0:      continue
            else:
                base = {1:10, 2:100, 3:1000, 4:100000}.get(cnt, 0)
                if ends == 1: base //= 4
            score += base if mine else -float(base * 2.0)
    return score

# =====================================================================
# 2. CUSTOM CNN
# =====================================================================
class CaroCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=512):
        super().__init__(observation_space, features_dim)
        in_ch = observation_space.shape[0]  
        self.cnn = nn.Sequential(
            nn.Conv2d(in_ch, 64,  kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64,    64,  kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64,    128, kernel_size=3, stride=2, padding=1), nn.ReLU(), 
            nn.Conv2d(128,   128, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((8, 8)),
            nn.Flatten(),
        )
        with torch.no_grad():
            n_flatten = self.cnn(
                torch.as_tensor(observation_space.sample()[None]).float()
            ).shape[1]
        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())

    def forward(self, obs):
        return self.linear(self.cnn(obs.float()))

# =====================================================================
# 3. ENVIRONMENT V12.5
# =====================================================================
_RADIUS          = 3
_DILATION_STRUCT = np.ones((2*_RADIUS+1, 2*_RADIUS+1), dtype=bool)

REWARD_WIN   =  1.0
REWARD_LOSE  = -1.0
REWARD_BLOCK =  0.05
REWARD_SCALE =  1e-4

STEP_PENALTY = -0.005  

class CaroEnv(gym.Env):
    def __init__(self, opponent_level=0):
        super().__init__()
        self.opponent_level    = opponent_level  
        self.action_space      = spaces.Discrete(BOARD_SIZE * BOARD_SIZE)
        self.observation_space = spaces.Box(
            0, 1, shape=(3, BOARD_SIZE, BOARD_SIZE), dtype=np.uint8
        )
        self.agent_player = 1
        self.bot_player   = 2

    def set_opponent_level(self, level):
        self.opponent_level = level
        self.last_score = 0.0  

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.board         = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=np.int8)
        self.move_count    = 0
        self.active_cells  = set() 
        self.last_score    = 0.0
        self.last_opp_move = None
        
        self.agent_player = np.random.choice([1, 2])
        self.bot_player   = 2 if self.agent_player == 1 else 1

        if self.bot_player == 1:
            center = BOARD_SIZE // 2
            self.board[center, center] = self.bot_player
            self.active_cells.add((center, center))
            self.last_opp_move = center * BOARD_SIZE + center
            self.move_count += 1

        return self._obs(), {}

    def _obs(self):
        obs = np.zeros((3, BOARD_SIZE, BOARD_SIZE), dtype=np.uint8)
        obs[0] = (self.board == self.agent_player)
        obs[1] = (self.board == self.bot_player)
        if self.last_opp_move is not None:
            r, c = divmod(self.last_opp_move, BOARD_SIZE)
            obs[2, r, c] = 1
        return obs

    def _is_winning_move(self, player, r, c):
        if self.board[r, c] != 0: return False
        self.board[r, c] = player
        result = check_win(self.board, r, c, player)
        self.board[r, c] = 0
        return result

    def _get_opponent_move(self):
        masks = self.action_masks()
        valid_actions = np.where(masks)[0]
        
        if len(valid_actions) == 0:
            return BOARD_SIZE * BOARD_SIZE // 2 + BOARD_SIZE // 2

        lv = self.opponent_level

        if lv >= 1:
            for a in valid_actions:
                r, c = divmod(a, BOARD_SIZE)
                if self._is_winning_move(self.bot_player, r, c): return a

        if lv >= 3:
            for a in valid_actions:
                r, c = divmod(a, BOARD_SIZE)
                if self._is_winning_move(self.agent_player, r, c): return a

        random_prob = max(0.0, 1.0 - lv * 0.15)  
        if np.random.rand() < random_prob:
            return np.random.choice(valid_actions)

        candidates = valid_actions if len(valid_actions) <= 40 else \
                     np.random.choice(valid_actions, 40, replace=False)
                     
        best_score, best_a = -float('inf'), candidates[0]
        for a in candidates:
            r, c = divmod(a, BOARD_SIZE)
            self.board[r, c] = self.bot_player
            score = static_evaluate(self.board, self.bot_player, self.active_cells | {(r, c)})
            self.board[r, c] = 0
            if score > best_score:
                best_score, best_a = score, a
                
        return best_a

    def step(self, action):
        x, y = divmod(action, BOARD_SIZE)

        if self.board[x, y] != 0:
            return self._obs(), REWARD_LOSE, True, False, {"is_success": False}

        is_blocking = self._is_winning_move(self.bot_player, x, y)

        self.board[x, y] = self.agent_player
        self.active_cells.add((x, y))
        self.move_count += 1

        if check_win(self.board, x, y, self.agent_player):
            return self._obs(), REWARD_WIN, True, False, {"is_success": True}
        if self.move_count >= MAX_MOVES:
            return self._obs(), 0.0, True, False, {"is_success": False}

        opp_a = self._get_opponent_move()
        self.last_opp_move = opp_a
        ox, oy = divmod(opp_a, BOARD_SIZE)
        self.board[ox, oy] = self.bot_player
        self.active_cells.add((ox, oy))
        self.move_count += 1

        if check_win(self.board, ox, oy, self.bot_player):
            return self._obs(), REWARD_LOSE, True, False, {"is_success": False}
        if self.move_count >= MAX_MOVES:
            return self._obs(), 0.0, True, False, {"is_success": False}

        block_bonus = REWARD_BLOCK if is_blocking else 0.0

        if self.opponent_level >= 7:
            shaped = 0.0
        elif self.opponent_level >= 5:
            fade   = max(0.0, 1.0 - (self.opponent_level - 5) * 0.4)
            scale  = REWARD_SCALE * fade
            cap    = max(0.05, 0.3 * fade)
            
            current_score   = static_evaluate(self.board, self.agent_player, self.active_cells)
            progress_reward = (current_score - self.last_score) * scale
            self.last_score = current_score
            shaped = float(np.clip(progress_reward + block_bonus * fade, -cap, cap))
        else:
            current_score   = static_evaluate(self.board, self.agent_player, self.active_cells)
            progress_reward = (current_score - self.last_score) * REWARD_SCALE
            self.last_score = current_score
            shaped = float(np.clip(progress_reward + block_bonus, -1.0, 1.0))

        return self._obs(), shaped + STEP_PENALTY, False, False, {}

    def action_masks(self):
        mask = np.zeros(BOARD_SIZE * BOARD_SIZE, dtype=bool)
        if not self.active_cells:
            for r in range(15, 25):
                for c in range(15, 25): mask[r * BOARD_SIZE + c] = True
            return mask

        occupied = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=bool)
        rows, cols = zip(*self.active_cells)
        occupied[list(rows), list(cols)] = True
        dilated = binary_dilation(occupied, structure=_DILATION_STRUCT)
        np.logical_and(dilated, self.board == 0, out=dilated)
        return dilated.ravel()

def make_env(opponent_level=0):
    def _init():
        env = CaroEnv(opponent_level=opponent_level)
        env = Monitor(env, info_keywords=("is_success",))
        return env
    return _init

# =====================================================================
# 4. CALLBACKS
# =====================================================================
CSV_HEADER = [
    "timestep", "mean_reward", "mean_ep_len", "win_rate",
    "policy_loss", "value_loss", "entropy_loss",
    "approx_kl", "clip_fraction", "fps", "elapsed_min", "opp_level",
    "learning_rate", "ent_coef"  
]

class TrainingLogger(BaseCallback):
    def __init__(self, csv_path, log_freq=1, verbose=1): 
        super().__init__(verbose)
        self.csv_path = csv_path
        self.log_freq = log_freq
        self._count   = 0
        self._t0      = time.time()
        if not os.path.exists(csv_path):
            with open(csv_path, "w", newline="") as f:
                csv.writer(f).writerow(CSV_HEADER)

    def _get_metric(self, key, default=None):
        try:
            v = self.logger.name_to_value.get(key, default)
            return float(v) if v is not None else default
        except Exception:
            return default

    def _on_rollout_end(self):
        self._count += 1
        if self._count % self.log_freq != 0: return
        buf = self.model.ep_info_buffer
        elapsed = max(time.time() - self._t0, 1)
        
        current_level = self.training_env.get_attr("opponent_level")[0]
        current_lr = self.model.policy.optimizer.param_groups[0]["lr"]
        current_ent = self.model.ent_coef

        row = [
            self.num_timesteps,
            round(float(np.mean([e.get("r",0) for e in buf])) if buf else 0, 6),
            round(float(np.mean([e.get("l",0) for e in buf])) if buf else 0, 2),
            round(sum(1 for e in buf if e.get("is_success")) / len(buf) if buf else 0, 4),
            self._get_metric("train/policy_gradient_loss"),
            self._get_metric("train/value_loss"),
            self._get_metric("train/entropy_loss"),
            self._get_metric("train/approx_kl"),
            self._get_metric("train/clip_fraction"),
            int(self.num_timesteps / elapsed),
            round(elapsed / 60, 2),
            current_level,
            current_lr,     
            current_ent      
        ]
        with open(self.csv_path, "a", newline="") as f:
            csv.writer(f).writerow(row)
        if self.verbose >= 1:
            print(
                f"  [V12.5-L{current_level}] step={self.num_timesteps:>10,} | "
                f"rew={row[1]:>8.5f} | win={row[3]*100:>5.1f}% | "
                f"kl={row[7] or 0:.2e} | lr={current_lr:.2e} | "
                f"fps={row[9]:>5} | t={row[10]:.1f}min"
            )

    def _on_step(self): return True

class EntropyAnnealCallback(BaseCallback):
    def __init__(self, total_steps, ent_start=0.05, ent_end=0.005, verbose=1):
        super().__init__(verbose)
        self.total_steps = total_steps
        self.ent_start   = ent_start
        self.ent_end     = ent_end

    def _on_step(self):
        progress = min(self.num_timesteps / self.total_steps, 1.0)
        new_ent  = self.ent_start + progress * (self.ent_end - self.ent_start)
        self.model.ent_coef = new_ent
        return True

class AdaptiveLRCallback(BaseCallback):
    def __init__(self, kl_threshold=0.03, lr_factor=0.5, min_lr=1e-5, verbose=1):
        super().__init__(verbose)
        self.kl_threshold = kl_threshold
        self.lr_factor    = lr_factor
        self.min_lr       = min_lr

    def _on_step(self): return True 

    def _on_rollout_end(self):
        try:
            kl = float(self.logger.name_to_value.get("train/approx_kl", 0))
        except Exception:
            return True

        actual_lr = self.model.policy.optimizer.param_groups[0]["lr"]

        if kl > self.kl_threshold and actual_lr > self.min_lr:
            new_lr = max(actual_lr * self.lr_factor, self.min_lr)
            
            for pg in self.model.policy.optimizer.param_groups:
                pg["lr"] = new_lr
                
            self.model.learning_rate = new_lr 
            
            if self.verbose:
                print(f"   [🔧 AdaptiveLR] KL Divergence ({kl:.3f}) VƯỢT NGƯỠNG AN TOÀN! "
                      f"→ Giảm Learning Rate xuống: {new_lr:.2e}")
        return True

class CurriculumCallback(BaseCallback):
    def __init__(self, 
                 upgrade_thresholds=None, 
                 min_steps=300_000, 
                 patience=5,          
                 collapse_threshold=500_000,
                 verbose=1):
        super().__init__(verbose)
        self.thresholds = upgrade_thresholds or {0: 0.45} 
        self.min_steps           = min_steps
        self.patience            = patience
        self.collapse_threshold  = collapse_threshold  
        
        self.current_level = 0
        self.max_level     = 10  
        
        self._step_at_level_start   = 0  
        self._patience_counter      = 0
        self._best_winrate_at_level = 0.0

    def _get_metrics(self):
        buf = self.model.ep_info_buffer
        if not buf: return 0.0, 0.0
        win_rate = sum(1 for e in buf if e.get("is_success")) / len(buf)
        mean_reward = float(np.mean([e.get("r", 0) for e in buf]))
        return win_rate, mean_reward

    def _on_step(self): return True

    def _on_rollout_end(self):
        if self.current_level >= self.max_level:
            return True

        steps_at_level = self.num_timesteps - self._step_at_level_start
        buf = self.model.ep_info_buffer

        if len(buf) < 100:
            return True

        win_rate, mean_reward = self._get_metrics()
        threshold = self.thresholds.get(self.current_level, 0.45) 

        # Tăng ngưỡng rollback lên 0.10 để chặn việc lãng phí compute
        if (self.current_level >= 2 
                and steps_at_level > self.collapse_threshold 
                and win_rate < 0.10):
                
            self.current_level -= 1
            self._step_at_level_start     = self.num_timesteps
            self._patience_counter        = 0
            self._best_winrate_at_level   = 0.0
            
            self.training_env.env_method("set_opponent_level", self.current_level)
            self.model.ep_info_buffer.clear()
            
            if self.verbose:
                print(f"\n{'='*58}")
                print(f"🚨 CẢNH BÁO: Mắc kẹt ở Level {self.current_level + 1} quá lâu với Win Rate {win_rate:.1%}")
                print(f"   => Khởi động chiến dịch ROLLBACK. Hạ cấp đối thủ về Level {self.current_level}")
                print(f"{'='*58}\n")
            return True

        if steps_at_level < self.min_steps:
            return True

        self._best_winrate_at_level = max(self._best_winrate_at_level, win_rate)
        
        is_mastering = (win_rate >= threshold) and (mean_reward > -0.6)

        if is_mastering:
            self._patience_counter += 1  
            if self.verbose:
                print(f"   [⏳ Patience Check: {self._patience_counter}/{self.patience}] "
                      f"Win={win_rate:.1%} (Cần >= {threshold:.1%}) | Rew={mean_reward:.2f}")
        else:
            if self._patience_counter > 0 and self.verbose:
                print(f"   [📉 Gãy chuỗi phong độ! Đặt lại Patience về 0]")
            self._patience_counter = 0

        if self._patience_counter >= self.patience:
            self.current_level += 1
            self._step_at_level_start     = self.num_timesteps  
            self._patience_counter        = 0 
            self._best_winrate_at_level   = 0.0
            
            self.training_env.env_method("set_opponent_level", self.current_level)
            self.model.ep_info_buffer.clear()
            
            # Boost Entropy khi vào vùng Sparse Reward
            if self.current_level >= 7:
                self.model.ent_coef = max(self.model.ent_coef, 0.02)
                if self.verbose:
                    print(f"   [🔥 Entropy Boost] Vào sparse zone, ent_coef -> {self.model.ent_coef:.3f}")

            if self.verbose:
                print(f"\n{'='*58}")
                print(f"🎓 LỄ TỐT NGHIỆP: Agent thăng tiến từ Level {self.current_level-1} lên Level {self.current_level} "
                      f"tại step {self.num_timesteps:,}")
                print(f"{'='*58}\n")
                
        return True

class SaveCurriculumStateCallback(BaseCallback):
    def __init__(self, curriculum_cb, state_path, save_freq=32_768):
        super().__init__()
        self.curriculum_cb = curriculum_cb
        self.state_path    = state_path
        self.save_freq     = save_freq

    def _on_step(self): return True

    def _on_rollout_end(self):
        if self.num_timesteps % self.save_freq < 32_768:
            np.save(self.state_path, {
                "level":                self.curriculum_cb.current_level,
                "step_at_level_start":  self.curriculum_cb._step_at_level_start,
                "steps_done":           self.num_timesteps,
            })
        return True

# =====================================================================
# 5. VẼ BIỂU ĐỒ 
# =====================================================================
def plot_training(csv_path, out_png, smooth=15):
    import pandas as pd
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    from matplotlib.ticker import FuncFormatter

    df = pd.read_csv(csv_path)
    for col in df.columns: df[col] = pd.to_numeric(df[col], errors="coerce")
    if len(df) == 0: return

    def smth(col):
        s = df[col].dropna()
        return s.index, s, s.rolling(smooth, min_periods=1, center=True).mean()
    def fmt_M(x, _):
        if x>=1e6: return f"{x/1e6:.1f}M"
        if x>=1e3: return f"{x/1e3:.0f}K"
        return str(int(x))

    COLORS = {"reward":"#378ADD","win":"#1D9E75","ep_len":"#BA7517",
              "pol":"#D85A30","val":"#7F77DD","ent":"#888780",
              "kl":"#D4537E","clip":"#D85A30","fps":"#444441","raw":"#D3D1C7",
              "lr":"#2A9D8F", "ent_c":"#E76F51"}
              
    specs  = [
        ("mean_reward","Mean reward","reward"), ("win_rate","Win rate (%)","win"),
        ("mean_ep_len","Episode length","ep_len"), ("policy_loss","Policy loss","pol"),
        ("value_loss","Value loss","val"), ("entropy_loss","Entropy loss","ent"),
        ("approx_kl","Approx KL","kl"), ("clip_fraction","Clip fraction","clip"),
        ("fps","FPS","fps"),
        ("learning_rate", "Learning Rate", "lr"),  
        ("ent_coef", "Entropy Coef", "ent_c")      
    ]
    
    fig = plt.figure(figsize=(16,16)); fig.patch.set_facecolor("#FAFAF8")
    gs  = gridspec.GridSpec(4,3,figure=fig,hspace=0.48,wspace=0.38,
                            left=0.07,right=0.96,top=0.91,bottom=0.07)
    x   = df["timestep"]

    for idx,(col,title,ck) in enumerate(specs):
        ax = fig.add_subplot(gs[idx//3, idx%3])
        ax.set_facecolor("#F7F5F0")
        ax.set_title(title,fontsize=10,fontweight="bold",color="#2C2C2A",pad=5)
        ax.tick_params(labelsize=7,colors="#5F5E5A")
        ax.xaxis.set_major_formatter(FuncFormatter(fmt_M))
        for sp in ax.spines.values(): sp.set_edgecolor("#D3D1C7")
        ax.grid(True,alpha=0.35,color="#D3D1C7",linewidth=0.6)
        if col not in df.columns or df[col].dropna().empty: continue
        ix,y_raw,y_sm = smth(col); xi = x.loc[ix]
        
        if col=="win_rate": y_raw,y_sm = y_raw*100,y_sm*100; ax.set_ylim(0,100)
        if col=="clip_fraction":
            ax.axhline(0.1,color="#1D9E75",lw=1,ls="--",alpha=0.7,label="~0.1")
            ax.legend(fontsize=6)
        if col=="approx_kl":
            ax.axhline(0.03,color="#D4537E",lw=1,ls="--",alpha=0.7,label="0.03 (Danger)")
            ax.legend(fontsize=6)
        if col=="learning_rate":
            ax.set_yscale('log')
            
        ax.plot(xi,y_raw,color=COLORS["raw"],alpha=0.3,lw=0.7,zorder=1)
        ax.plot(xi,y_sm, color=COLORS[ck],  lw=1.8,    zorder=2)

    total_steps = int(x.max())
    total_min   = df["elapsed_min"].max() if "elapsed_min" in df.columns else 0
    fig.suptitle(f"Caro RL V12.5 (Ultimate + Resume Support) — {fmt_M(total_steps,None)} steps ({total_min:.0f} min)",
                 fontsize=12,fontweight="bold",color="#2C2C2A",y=0.965)
    plt.savefig(out_png,dpi=150,bbox_inches="tight",facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"Đã lưu: {out_png}")

# =====================================================================
# 6. TRAINING CHÍNH — CÓ RESUME MODEL VÀ NỐI TIẾP CSV (KAGGLE SAFE)
# =====================================================================
if __name__ == "__main__":
    import shutil # <--- Nhớ import cái này ở đầu file nếu chưa có

    TOTAL_STEPS = 25_000_000
    NUM_ENVS    = 16
    SAVE_DIR    = "/kaggle/working/ppo_caro_v12/"
    
    # --- ĐƯỜNG DẪN INPUT (READ-ONLY) ---
    INPUT_MODEL_PATH = "/kaggle/input/notebooks/trubui/quintetx-rl-ppo-277cf3/ppo_caro_v12/caro_v12_13800000_steps.zip"
    INPUT_CSV_PATH   = "/kaggle/input/notebooks/trubui/quintetx-rl-ppo-277cf3/ppo_caro_v12/training_log.csv"
    
    # --- ĐƯỜNG DẪN OUTPUT (WORKING DIR) ---
    MODEL_PATH  = os.path.join(SAVE_DIR, "caro_v12.zip")
    CSV_PATH    = os.path.join(SAVE_DIR, "training_log.csv")
    PLOT_PATH   = os.path.join(SAVE_DIR, "training_report.png")
    STATE_PATH  = os.path.join(SAVE_DIR, "curriculum_state.npy")

    os.makedirs(SAVE_DIR, exist_ok=True)

    # ── LOGIC NỐI TIẾP CSV TỪ KAGGLE INPUT ──
    if os.path.exists(INPUT_CSV_PATH):
        if not os.path.exists(CSV_PATH):
            # Nếu trong Working chưa có file CSV nào, ta bốc nguyên file cũ qua
            print("📄 Đang copy file training_log.csv cũ sang thư mục Working để viết tiếp...")
            shutil.copy2(INPUT_CSV_PATH, CSV_PATH)
        else:
            print("📄 File log đã tồn tại ở Working, sẽ nối tiếp vào file này.")

    env = SubprocVecEnv([make_env(opponent_level=0) for _ in range(NUM_ENVS)])

    policy_kwargs = dict(
        normalize_images=False,
        features_extractor_class=CaroCNN,
        features_extractor_kwargs=dict(features_dim=512),
        net_arch=dict(pi=[256, 128], vf=[256, 128])
    )

    curriculum_cb = CurriculumCallback(
        min_steps=300_000, 
        patience=5,
        collapse_threshold=500_000, 
        verbose=1
    )

    # ── RESUME MODEL LOGIC ──
    if os.path.exists(INPUT_MODEL_PATH):
        print(f"♻️ TÌM THẤY BÁU VẬT 13.8M STEPS TẠI KAGGLE INPUT!")
        model = MaskablePPO.load(INPUT_MODEL_PATH, env=env, device="cuda")
        model.ep_info_buffer = deque(maxlen=300)

        # Trạng thái Hardcode (Vì file cũ không có file .npy)
        current_level_hardcoded = 9
        steps_done_hardcoded    = 13_800_000
        
        curriculum_cb.current_level          = current_level_hardcoded
        curriculum_cb._step_at_level_start   = steps_done_hardcoded
        curriculum_cb._patience_counter      = 0  
        
        env.env_method("set_opponent_level", curriculum_cb.current_level)
        
        remaining = max(TOTAL_STEPS - steps_done_hardcoded, 1_000_000)
        reset_timesteps = False # Để biểu đồ vẽ nối tiếp mượt mà
        
        # Lưu file state cho lần sập Kaggle sau
        np.save(STATE_PATH, {
            "level": current_level_hardcoded,
            "step_at_level_start": steps_done_hardcoded,
            "steps_done": steps_done_hardcoded,
        })
        
        print(f"   Level: {curriculum_cb.current_level} | Hoàn thành: {steps_done_hardcoded:,} | Cần cày: {remaining:,}")
    else:
        print("❌ LỖI: Không tìm thấy model tại Input!")
        exit()

    print(f"Policy params: {sum(p.numel() for p in model.policy.parameters()):,}")

    callbacks = CallbackList([
        CheckpointCallback(
            save_freq=max(100_000 // NUM_ENVS, 1),
            save_path=SAVE_DIR, 
            name_prefix="caro_v12"
        ),
        TrainingLogger(csv_path=CSV_PATH, log_freq=1, verbose=1),
        EntropyAnnealCallback(TOTAL_STEPS, ent_start=0.05, ent_end=0.005),
        AdaptiveLRCallback(kl_threshold=0.03, lr_factor=0.5, min_lr=1e-5),
        curriculum_cb,
        SaveCurriculumStateCallback(curriculum_cb, STATE_PATH),
    ])

    try:
        model.learn(
            total_timesteps     = remaining,
            callback            = callbacks,
            reset_num_timesteps = reset_timesteps,
        )
    except KeyboardInterrupt:
        print("\n🛑 Dừng tay.")
    finally:
        model.save(MODEL_PATH)
        print(f"💾 Saved: {MODEL_PATH}")
        env.close()
        if os.path.exists(CSV_PATH):
            plot_training(CSV_PATH, PLOT_PATH)

2026-05-06 16:08:18.018376: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778083698.419677      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778083698.536852      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778083699.542883      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778083699.542926      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778083699.542929      23 computation_placer.cc:177] computation placer alr

📄 Đang copy file training_log.csv cũ sang thư mục Working để viết tiếp...


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
2026-05-06 16:09:10.529865: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778083750.696798      69 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-06 16:09:10.740576: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT

♻️ TÌM THẤY BÁU VẬT 13.8M STEPS TẠI KAGGLE INPUT!
   Level: 9 | Hoàn thành: 13,800,000 | Cần cày: 11,200,000
Policy params: 4,989,953
  [V12.5-L9] step=13,832,768 | rew=-0.71453 | win= 16.0% | kl=0.00e+00 | lr=1.00e-04 | fps=185268 | t=1.2min
  [V12.5-L9] step=13,865,536 | rew=-0.73413 | win= 15.0% | kl=3.05e-02 | lr=1.00e-05 | fps=75088 | t=3.1min
  [V12.5-L9] step=13,898,304 | rew=-0.78507 | win= 12.3% | kl=2.66e-03 | lr=1.00e-05 | fps=46988 | t=4.9min
  [V12.5-L9] step=13,931,072 | rew=-0.72083 | win= 15.7% | kl=2.00e-03 | lr=1.00e-05 | fps=34109 | t=6.8min
  [V12.5-L9] step=13,963,840 | rew=-0.72082 | win= 15.7% | kl=2.47e-03 | lr=1.00e-05 | fps=26711 | t=8.7min
  [V12.5-L9] step=13,996,608 | rew=-0.81732 | win= 10.7% | kl=1.59e-03 | lr=1.00e-05 | fps=21991 | t=10.6min
  [V12.5-L9] step=14,029,376 | rew=-0.70107 | win= 16.7% | kl=1.42e-03 | lr=1.00e-05 | fps=18665 | t=12.5min
  [V12.5-L9] step=14,062,144 | rew=-0.81125 | win= 11.0% | kl=1.87e-03 | lr=1.00e-05 | fps=16246 | t=14.4mi